# 🐼 Pandas Advanced (Part 2) — Complete Masterclass
> **Course Track:** Prime AI/ML Batch  
> **Topic:** Data Transformation, GroupBy & Aggregations, Melt & Pivot, Merging & Joins, Concatenation & Visualization  
> **Language / Notes Mode:** Easy Natural Hinglish  

---

### 📌 Pandas Part 2 Me Hum Kya Seekhenge?
Part 1 me humne data load karna, inspect karna aur clean karna seekha. Part 2 me hum real-world **Feature Engineering** aur **Data Aggregation** seekhenge:
1. **Transformations:** `.apply()`, `.map()`, `.assign()`, `.replace()`.
2. **Sorting & Indexing:** Multi-level sorting, index reset, aur ranking.
3. **GroupBy & Aggregations:** Split-Apply-Combine pattern, custom aggregations.
4. **Reshaping:** Wide to Long (`melt`) aur Long to Wide (`pivot`).
5. **Merging & Joins:** SQL-like Inner, Left, Right, Outer joins.
6. **Concatenation:** Tables ko vertical ya horizontal jodna (`pd.concat`).
7. **Basic Visualization:** Histograms aur Scatter plots directly with Pandas.


## 🔄 Operation 1: Data Transformation (`apply`, `map`, `assign`, `replace`)

### 💡 Kyu Zaroorat Hai?
Feature engineering me existing columns se naye meaningful features calculate karne padte hain (jaise income par tax calculate karna, ya categorical codes ko readable labels me badalna).

### 🔍 Methods:
- `.apply(fn)`: Custom function ya lambda ko column ke har element par apply karta hai.
- `.map(dict)`: Values ko dictionary mapping ke through replace karta hai.
- `.assign(new_col=...)`: Ek ya zyada naye columns chain karke create karta hai.
- `.replace(old, new)`: Specific values ko swap karta hai.


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("employee_data.csv")
print("Original Employee Table:\n", df)

# 1. .apply() with Lambda (Tax Bracket calculation)
# Agar Salary > 70000 -> 20% Tax, warna 10% Tax
df["Tax_Bracket"] = df["Salary"].apply(lambda sal: "20%" if sal > 70000 else "10%")

# 2. .map() with Dictionary (Department short codes)
dept_codes = {"HR": "Human Resources", "IT": "Information Tech", "Finance": "Fin", "Marketing": "Mkt"}
df["Dept_Full"] = df["Department"].map(dept_codes)

# 3. .assign() (New column Bonus: 10% of salary)
df = df.assign(Bonus=df["Salary"] * 0.10)

# 4. .replace() (Specific values replace karna)
df["Department"] = df["Department"].replace({"HR": "People Ops"})

print("\n--- Transformed Employee Table ---\n", df[["Name", "Department", "Salary", "Bonus", "Tax_Bracket", "Dept_Full"]])


## 🔢 Operation 2: Sorting, Renaming, Index Resetting & Ranking

### 💡 Kyu Zaroorat Hai?
Reports generate karte waqt data ko highest-to-lowest salary me sort karna, columns ke readable names rakhna, aur ranks dena zaroori hota hai.

### 🔍 Key Functions:
- `.rename(columns={...})`: Columns ka naam badalna.
- `.sort_values(by=..., ascending=...)`: Values ke basis par sort karna (single ya multiple columns).
- `.reset_index(drop=True)`: Sort karne ke baad messy index ko wapis 0, 1, 2... se reset karna.
- `.rank(method="dense", ascending=False)`: Highest value ko rank 1 dena.


In [ ]:
# 1. Renaming Columns
df_renamed = df.rename(columns={"Salary": "Base_Salary", "Bonus": "Annual_Bonus"})

# 2. Multi-column Sorting (Department alphabetically, then Base_Salary descending)
sorted_df = df_renamed.sort_values(by=["Department", "Base_Salary"], ascending=[True, False])
print("--- Sorted by Department & Salary (Messy Index) ---\n", sorted_df[["Name", "Department", "Base_Salary"]].head(5))

# 3. Reset Index
clean_sorted = sorted_df.reset_index(drop=True)
print("\n--- Index Reset (Clean 0, 1, 2...) ---\n", clean_sorted[["Name", "Department", "Base_Salary"]].head(5))

# 4. Ranking (Highest salary gets Rank 1)
clean_sorted["Salary_Rank"] = clean_sorted["Base_Salary"].rank(ascending=False, method="dense").astype(int)
print("\n--- Salary Ranks (Top 5) ---\n", clean_sorted[["Salary_Rank", "Name", "Base_Salary"]].sort_values("Salary_Rank").head(5))


## 🔀 Operation 3: Reordering & Column Arrangement

### 💡 Kyu Zaroorat Hai?
Presentation ya machine learning pipelines me target column (label) ko aakhri me shift karna ya identifier (ID) ko first position par rakhna best practice hoti hai.


In [ ]:
# Column order rearrange karna
desired_order = ["ID", "Name", "Department", "Age", "Base_Salary", "Salary_Rank"]
reordered_df = clean_sorted[desired_order]
print("Reordered DataFrame:\n", reordered_df.head(4))


## 👥 Operation 4: GroupBy & Summary Aggregations (Split-Apply-Combine)

### 💡 Kyu Zaroorat Hai? (Data Analytics ka Heart!)
SQL me jaise `GROUP BY` hota hai, waise hi Pandas me data ko categories me split karke summary metrics (Average, Sum, Count, Max) nikalte hain.

### 🔍 Key Techniques:
- `df.groupby("Department")["Salary"].mean()`: Single column aggregation.
- `.agg(["mean", "min", "max", "count"])`: Multiple statistical functions ek sath apply karna.
- **Named Aggregations:** `agg(avg_sal=("Salary", "mean"), total_emp=("ID", "count"))` — Isse output table ke columns clean aur professional lagte hain.


In [ ]:
# 1. Simple GroupBy: Department-wise Average Salary
dept_avg_sal = df.groupby("Department")["Salary"].mean()
print("Department-wise Average Salary:\n", dept_avg_sal)

# 2. Multiple Aggregations on Single Column
dept_stats = df.groupby("Department")["Salary"].agg(["count", "mean", "min", "max"])
print("\n--- Department-wise Salary Summary Statistics ---\n", dept_stats)

# 3. Named Aggregations on Multiple Columns (Clean Columns)
custom_summary = df.groupby("Department").agg(
    total_staff=("ID", "count"),
    avg_age=("Age", "mean"),
    avg_salary=("Salary", "mean"),
    max_salary=("Salary", "max")
).round(1)

print("\n--- Professional Named Aggregation Table ---\n", custom_summary)


## 🍦 Operation 5: Reshaping Data with `melt()` (Wide to Long Format)

### 💡 Kyu Zaroorat Hai?
Spreadsheets me aksar data **Wide format** me hota hai (e.g., Har saal ya metric ka alag column). Lekin visualization libraries (jaise Seaborn, PowerBI) aur ML models ko **Long/Tidy format** chahiye hota hai.

### 🔍 `melt()` Parameters:
- `id_vars`: Wo columns jinhe fix (identifier) rakhna hai.
- `value_vars`: Wo columns jinhe unpivot karke rows me convert karna hai.
- `var_name`: Naye category column ka naam.
- `value_name`: Naye numbers wale column ka naam.


In [ ]:
# Wide format company financial data
finance_wide = pd.DataFrame({
    "Country": ["USA", "USA", "India", "India"],
    "Year": [2023, 2024, 2023, 2024],
    "Sales_Cr": [150, 180, 120, 160],
    "Profit_Cr": [30, 42, 25, 38]
})

print("--- Original Wide Format Table ---\n", finance_wide)

# Melt into Long format
finance_long = finance_wide.melt(
    id_vars=["Country", "Year"],
    value_vars=["Sales_Cr", "Profit_Cr"],
    var_name="Financial_Metric",
    value_name="Amount_in_Crores"
)

print("\n--- Melted Long / Tidy Format Table ---\n", finance_long)


## 🔁 Operation 6: Reshaping Data with `pivot()` (Long to Wide Format)

### 💡 Kyu Zaroorat Hai?
Long format data ko summarize karke cross-tabulation table ya dashboard matrix banane ke liye `pivot()` use karte hain.

### 🔍 `pivot()` Parameters:
- `index`: Nayi table ke row labels kaunse columns banenge.
- `columns`: Kis column ke unique values nayi table ke column headers banenge.
- `values`: Table ke cells me kaunse numbers fill honge.


In [ ]:
# Long format ko wapis wide me pivot karna
pivoted_table = finance_long.pivot(
    index=["Country", "Year"],
    columns="Financial_Metric",
    values="Amount_in_Crores"
).reset_index()

print("--- Pivoted Back to Wide Format ---\n", pivoted_table)


## 🔗 Operation 7: Merging & Joining DataFrames (Inner, Left, Right, Outer)

### 💡 Kyu Zaroorat Hai?
Real databases relational hote hain. Customers ka data alag table me hota hai aur unke Orders ka data alag table me. Dono ko common key (`customer_id`) ke basis par join karna padta hai.

### 🔍 4 Types of Joins (`how=` parameter):
1. **`inner` (Default):** Sirf wahi records aate hain jo dono tables me present hon.
2. **`left`:** Left table ke saare records aayenge; right table me match na hone par `NaN` aayega.
3. **`right`:** Right table ke saare records aayenge.
4. **`outer`:** Dono tables ke saare records aayenge; jaha match nahi hoga waha `NaN` lagega.


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Aarav", "Isha", "Rohan", "Sneha"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104],
    "customer_id": [2, 1, 4, 5],      # Notice customer 5 is new, customer 3 has no order
    "amount": [4500, 1200, 8900, 3100]
})

print("Customers Table:\n", customers)
print("\nOrders Table:\n", orders)

# 1. Inner Join (Only matching customers: 1, 2, 4)
inner_join = pd.merge(customers, orders, on="customer_id", how="inner")
print("\n--- Inner Join (Common in Both) ---\n", inner_join)

# 2. Left Join (All customers kept; Rohan gets NaN for order)
left_join = pd.merge(customers, orders, on="customer_id", how="left")
print("\n--- Left Join (Keep All Customers) ---\n", left_join)

# 3. Outer Join (All customers + all orders kept)
outer_join = pd.merge(customers, orders, on="customer_id", how="outer")
print("\n--- Outer Join (Full Union with NaNs) ---\n", outer_join)


## 🧱 Operation 8: Data Concatenation (`pd.concat` Row-wise & Column-wise)

### 💡 Kyu Zaroorat Hai?
Jab alag-alag months ya branches ka data same structure me aaye, toh unhe ek ke niche ek stack karne ke liye `pd.concat` use hota hai.

### 🔍 Axes in Concat:
- `axis=0` (Default): Vertical stacking (Rows ek ke niche ek judti hain). `ignore_index=True` se fresh index banta hai.
- `axis=1`: Horizontal stacking (Columns barabar me judte hain).


In [ ]:
batch_jan = pd.DataFrame({
    "id": [1, 2],
    "student": ["Karan", "Simran"],
    "score": [82, 89]
})

batch_feb = pd.DataFrame({
    "id": [3, 4],
    "student": ["Vikas", "Pooja"],
    "score": [75, 94]
})

# Vertical Concatenation (Row-wise)
combined_students = pd.concat([batch_jan, batch_feb], ignore_index=True)
print("--- Vertically Concatenated Batches (axis=0) ---\n", combined_students)

# Horizontal Concatenation (Column-wise)
extra_info = pd.DataFrame({
    "city": ["Delhi", "Mumbai", "Pune", "Jaipur"]
})
combined_with_city = pd.concat([combined_students, extra_info], axis=1)
print("\n--- Horizontally Combined with Cities (axis=1) ---\n", combined_with_city)


## 📈 Operation 9: Basic Data Visualization with Pandas

### 💡 Kyu Zaroorat Hai?
Bina kisi complex visualization library ke, Pandas ke paas directly `.hist()`, `.plot()`, aur `.boxplot()` methods hote hain jo data ka distribution turant visualize kar dete hain.


In [ ]:
# Pandas Built-in Plotting (Generates plot objects)
print("Employee Age Distribution (Summary Frequencies):")
print(df["Age"].value_counts().sort_index())

# Histogram of Salaries
# In Jupyter notebook, running df['Salary'].hist() directly displays the interactive graph
print("\nSalary Summary for Visualization:")
print(f"Min: {df['Salary'].min()} | Max: {df['Salary'].max()} | Mean: {df['Salary'].mean():.1f}")


## 💾 Operation 10: Exporting Processed Data to CSV & JSON

### 💡 Kyu Zaroorat Hai?
Transformation pipeline complete hone ke baad processed data ko clean CSV ya JSON files me serialize karna padta hai.


In [ ]:
# Final export
reordered_df.to_csv("final_processed_employees.csv", index=False)
reordered_df.to_json("final_processed_employees.json", orient="records", indent=2)

print("Files 'final_processed_employees.csv' & 'final_processed_employees.json' created successfully.")
print("Verified file exists:", os.path.exists("final_processed_employees.csv"))
